In [ ]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn evaluate seqeval hazm
!pip install -q git+https://github.com/kmkurn/pytorch-crf.git

  Preparing metadata (setup.py) ... done


In [ ]:
!pip install git+https://github.com/kmkurn/pytorch-crf.git

  Cloning https://github.com/kmkurn/pytorch-crf.git to /tmp/pip-req-build-gutt2h8k
  Running command git clone --filter=blob:none --quiet https://github.com/kmkurn/pytorch-crf.git /tmp/pip-req-build-gutt2h8k
  Resolved https://github.com/kmkurn/pytorch-crf.git to commit 623e3402d00a2728e99d6e8486010d67c754267b
  Preparing metadata (setup.py) ... done


In [ ]:
# -*- coding: utf-8 -*-
"""
Pars-ABSA Production (Fixed + Optimized for Persian + T4)
- Aspect: ParsBERT + BiLSTM + CRF + LLRD + Class Weights (با bias روی emissions)
- Sentiment: Weighted CrossEntropy
- Fixes: non-contiguous tensors, JSONL parser, offset alignment, class-weight on CRF, BiLSTM bug
"""

import os
import re
import json
import shutil
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.nn import CrossEntropyLoss, BCEWithLogitsLoss

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification, DataCollatorWithPadding,
    EarlyStoppingCallback, set_seed
)
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from hazm import Normalizer
import numpy as np
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall
from transformers.modeling_outputs import SequenceClassifierOutput

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
RND = 42
set_seed(RND)
random.seed(RND)
np.random.seed(RND)
torch.manual_seed(RND)
torch.backends.cudnn.benchmark = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_NAME = "HooshvareLab/bert-base-parsbert-uncased"
MAX_LENGTH = 160
NUM_EPOCHS_ASPECT = 10
NUM_EPOCHS_SENTIMENT = 8
EARLY_STOPPING_PATIENCE = 3
USE_FP16 = torch.cuda.is_available()
USE_CRF = True
USE_BILSTM = True
USE_RULE_ENSEMBLE = True
NORMALIZE_TEXT = True
HEAD_LR_MULTIPLIER = 6
LLRD_DECAY = 0.9
CLASS_WEIGHT_CAP = 15.0

ASPECT_CKPT_DIR = "/content/tmp_ckpt/aspect"
SENTIMENT_CKPT_DIR = "/content/tmp_ckpt/sentiment"
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/extracted/pars_absa_models"
ASPECT_FINAL_DIR = os.path.join(DRIVE_ROOT, "aspect_model")
SENTIMENT_FINAL_DIR = os.path.join(DRIVE_ROOT, "sentiment_model")

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}
aspect_label2id = {"O": 0, "B-ASP": 1, "I-ASP": 2}
aspect_id2label = {v: k for k, v in aspect_label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "Tokenizer isn't fast, offset mapping won't work well"
persian_normalizer = Normalizer()

Device: cuda


### NORMALIZE + OFFSET

In [ ]:
def normalize_text_and_offsets(text, start, end):
    if not NORMALIZE_TEXT or text is None:
        return text, start, end
    if start is None or end is None or start < 0 or end < 0 or end > len(text):
        term = None
    else:
        term = text[start:end]

    norm_text = persian_normalizer.normalize(text)
    if not term:
        return norm_text, -1, -1

    norm_term = persian_normalizer.normalize(term)
    if not norm_term:
        return norm_text, -1, -1

    occurrences = [m.start() for m in re.finditer(re.escape(norm_term), norm_text)]
    if not occurrences:
        return norm_text, -1, -1

    if len(occurrences) == 1:
        idx = occurrences[0]
    else:
        approx_ratio = start / max(len(text), 1)
        approx_pos = approx_ratio * len(norm_text)
        idx = min(occurrences, key=lambda o: abs(o - approx_pos))

    return norm_text, idx, idx + len(norm_term)

### LOAD PARS-ABSA

In [ ]:
print("\n--- Loading Pars-ABSA ---")
raw = load_dataset("ParsiAI/Pars-ABSA")
df_train_full = raw["train"].to_pandas()
df_val_full = raw["validation"].to_pandas() if "validation" in raw else None
df_test_full = raw["test"].to_pandas() if "test" in raw else None

text_col, aspect_col, label_col, from_col, to_col = "text", "term", "polarity", "from", "to"

def build_pair_fields(df):
    df = df.copy()
    texts = df[text_col].astype(str).tolist()
    terms = df[aspect_col].astype(str).tolist()
    froms = df[from_col].tolist() if from_col in df else [None] * len(df)
    tos = df[to_col].tolist() if to_col in df else [None] * len(df)

    review_texts, asp_starts, asp_ends = [], [], []
    for text, term, f, t in zip(texts, terms, froms, tos):
        start = end = -1
        if f is not None and t is not None:
            try:
                f_i, t_i = int(f), int(t)
                if 0 <= f_i < t_i <= len(text) and text[f_i:t_i] == term:
                    start, end = f_i, t_i
            except (ValueError, TypeError):
                pass
        if start == -1:
            idx = text.find(term)
            if idx != -1:
                start, end = idx, idx + len(term)

        norm_text, norm_start, norm_end = normalize_text_and_offsets(text, start, end)
        review_texts.append(norm_text)
        asp_starts.append(norm_start)
        asp_ends.append(norm_end)

    df["review_text"] = review_texts
    df["asp_start"] = asp_starts
    df["asp_end"] = asp_ends
    return df

df_train_full = build_pair_fields(df_train_full)
if df_val_full is not None:
    df_val_full = build_pair_fields(df_val_full)
if df_test_full is not None:
    df_test_full = build_pair_fields(df_test_full)

df_train_full = df_train_full[df_train_full["asp_start"] != -1].reset_index(drop=True)


### SENTIPERS EXTRA

In [ ]:
SENTIPERS_DIR = "/content/SentiPers"
if not os.path.exists(SENTIPERS_DIR):
    os.system(f"git clone --depth 1 https://github.com/phosseini/SentiPers.git {SENTIPERS_DIR}")

EXTRA_DIR = os.path.join(SENTIPERS_DIR, "data", "extra")

def parse_sentipers_extra_file(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        lines = [l.rstrip("\n") for l in f]

    words, polarity = [], None

    def flush_sentence():
        if not words or polarity is None or polarity == "":
            return
        text = " ".join(w for w, _ in words)
        spans, cur_start, pos = [], None, 0
        for w, is_t in words:
            w_start, w_end = pos, pos + len(w)
            if is_t:
                if cur_start is None:
                    cur_start = w_start
                cur_end = w_end
            else:
                if cur_start is not None:
                    spans.append((cur_start, cur_end))
                    cur_start = None
            pos = w_end + 1
        if cur_start is not None:
            spans.append((cur_start, cur_end))
        for s, e in spans:
            rows.append({"text": text, "term": text[s:e], "asp_start": s, "asp_end": e, "polarity_raw": polarity})

    for line in lines:
        if not line.strip():
            continue
        if line.startswith("[") and line.endswith("]") and "\t" not in line:
            if "@@@" in line:
                flush_sentence()
                words, polarity = [], None
            else:
                polarity = line.replace("[", "").replace("]", "").strip()
            continue
        parts = line.split("\t")
        w = parts[1] if len(parts) > 1 else ""
        is_target = len(parts) >= 3 and parts[2] == "T"
        words.append((w, is_target))
    flush_sentence()
    return rows

extra_rows = []
if os.path.isdir(EXTRA_DIR):
    for fn in os.listdir(EXTRA_DIR):
        if fn.endswith(".xml"):
            extra_rows.extend(parse_sentipers_extra_file(os.path.join(EXTRA_DIR, fn)))

df_sentipers_extra = pd.DataFrame(extra_rows)
polarity_map = {"-2": "negative", "-1": "negative", "0": "neutral", "+1": "positive", "+2": "positive"}
if len(df_sentipers_extra) > 0:
    df_sentipers_extra = df_sentipers_extra[df_sentipers_extra["polarity_raw"].isin(polarity_map)].copy()
    df_sentipers_extra["polarity"] = df_sentipers_extra["polarity_raw"].map(polarity_map)
    norm_rows = []
    for _, r in df_sentipers_extra.iterrows():
        norm_text, norm_start, norm_end = normalize_text_and_offsets(r["text"], r["asp_start"], r["asp_end"])
        if norm_start == -1:
            continue
        norm_rows.append({
            "review_text": norm_text, "term": norm_text[norm_start:norm_end],
            "polarity": r["polarity"], "asp_start": norm_start, "asp_end": norm_end,
        })
    df_sentipers_extra = pd.DataFrame(norm_rows)
else:
    df_sentipers_extra = pd.DataFrame(columns=["review_text", "term", "polarity", "asp_start", "asp_end"])

### CUSTOM JSONL

In [ ]:
CUSTOM_JSONL_PATH = "/content/llm_synthetic_final.jsonl"

def _find_non_overlapping(text, term, used_spans):
    for m in re.finditer(re.escape(term), text):
        s, e = m.start(), m.end()
        if not any(not (e <= us or s >= ue) for us, ue in used_spans):
            return s, e
    return -1, -1

def load_custom_jsonl(path):
    rows = []
    if not os.path.exists(path):
        print(f"[custom_jsonl] فایل پیدا نشد: {path}")
        return pd.DataFrame(columns=["review_text", "term", "polarity", "asp_start", "asp_end", "category"])

    stats = {"total_lines": 0, "json_decode_fail": 0, "no_matching_key": 0,
              "total_examples": 0, "no_text_or_aspects": 0,
              "total_aspects": 0, "polarity_invalid": 0, "term_not_found": 0, "success": 0}

    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            stats["total_lines"] += 1
            try:
                outer = json.loads(line)
            except json.JSONDecodeError:
                stats["json_decode_fail"] += 1
                continue

            if "examples" in outer:
                examples = outer["examples"]
                category = outer.get("category")
            elif "raw" in outer:
                try:
                    inner = json.loads(outer["raw"])
                    examples = inner.get("examples", [])
                    category = outer.get("category") or inner.get("category")
                except (json.JSONDecodeError, TypeError):
                    stats["no_matching_key"] += 1
                    continue
            elif "review_text" in outer and "aspects" in outer:
                examples = [outer]
                category = outer.get("category")
            elif "review_text" in outer and "term" in outer and "polarity" in outer:
                examples = [{
                    "review_text": outer["review_text"],
                    "aspects": [{
                        "term": outer["term"],
                        "polarity": outer["polarity"],
                        "asp_start": outer.get("asp_start"),
                        "asp_end": outer.get("asp_end"),
                    }]
                }]
                category = outer.get("category")
            else:
                stats["no_matching_key"] += 1
                continue

            for ex in examples:
                stats["total_examples"] += 1
                text = str(ex.get("review_text", "")).strip()
                aspects = ex.get("aspects", [])
                if not text or not aspects:
                    stats["no_text_or_aspects"] += 1
                    continue

                used_spans = []
                for asp in aspects:
                    stats["total_aspects"] += 1
                    term = str(asp.get("term", "")).strip()
                    polarity = str(asp.get("polarity", "")).strip().lower()
                    if not term or polarity not in label2id:
                        stats["polarity_invalid"] += 1
                        continue

                    given_start = asp.get("asp_start")
                    given_end = asp.get("asp_end")
                    start = end = -1
                    if given_start is not None and given_end is not None:
                        try:
                            gs, ge = int(given_start), int(given_end)
                            if 0 <= gs < ge <= len(text) and text[gs:ge] == term:
                                start, end = gs, ge
                        except (ValueError, TypeError):
                            pass

                    if start == -1:
                        start, end = _find_non_overlapping(text, term, used_spans)

                    if start == -1:
                        stats["term_not_found"] += 1
                        continue

                    used_spans.append((start, end))
                    norm_text, norm_start, norm_end = normalize_text_and_offsets(text, start, end)
                    if norm_start == -1:
                        stats["term_not_found"] += 1
                        continue

                    stats["success"] += 1
                    rows.append({
                        "review_text": norm_text,
                        "term": norm_text[norm_start:norm_end],
                        "polarity": polarity,
                        "asp_start": norm_start,
                        "asp_end": norm_end,
                        "category": category
                    })

    print("=== LOAD STATS (custom jsonl) ===")
    for k, v in stats.items():
        print(f"{k}: {v}")
    print(f"Custom synthetic aspects loaded: {len(rows)}")
    return pd.DataFrame(rows)

df_custom_synthetic = load_custom_jsonl(CUSTOM_JSONL_PATH)

### COMBINE DATA

In [ ]:
try:
    train_df, val_df_from_train = train_test_split(
        df_train_full, test_size=0.2, random_state=RND, stratify=df_train_full[label_col]
    )
except ValueError:
    train_df, val_df_from_train = train_test_split(df_train_full, test_size=0.2, random_state=RND)

if df_val_full is not None:
    val_df = df_val_full
else:
    val_df, val_df_from_train = train_test_split(val_df_from_train, test_size=0.5, random_state=RND)

test_df = df_test_full if df_test_full is not None else val_df_from_train

base_cols = ["review_text", "term", label_col, "asp_start", "asp_end"]
train_df = train_df[base_cols].copy()

if len(df_sentipers_extra) > 0:
    extra_real = df_sentipers_extra.rename(columns={"polarity": label_col})[base_cols]
    train_df = pd.concat([train_df, extra_real], ignore_index=True)

if len(df_custom_synthetic) > 0:
    synth_df = df_custom_synthetic.rename(columns={"polarity": label_col})[base_cols]
    max_allowed = int(len(train_df) * 0.15)
    synth_to_use = synth_df.sample(min(len(synth_df), max_allowed), random_state=RND)
    train_df = pd.concat([train_df, synth_to_use], ignore_index=True)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for d in [train_df, val_df, test_df]:
    d["label_id"] = d[label_col].map(label2id)

print(f"Final Dataset Splits (aspect-level, for Sentiment) -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Sentiment label distribution (train):")
print(train_df[label_col].value_counts(normalize=True))

# ASPECT DETECTION

### AGGREGATION

<div dir="rtl">
تجمیع همه‌ی aspectهای هر review در یک نمونه
 مشکل اصلی قبلی: هر review به‌ازای هر aspect تکرار می‌شد و لیبل‌های متناقض می‌ساخت
 (یک کپی «موتور»=B-ASP، کپی دیگر همان جمله «موتور»=O). این باعث می‌شد CRF/مدل نتواند الگوی
 پایداری یاد بگیرد. اینجا تمام span های یک review را یکجا جمع می‌کنیم.
</div>

In [ ]:
from collections import defaultdict

def aggregate_aspects(df):
    grouped = defaultdict(set)
    for _, row in df.iterrows():
        if row["asp_start"] is not None and row["asp_start"] != -1:
            grouped[row["review_text"]].add((int(row["asp_start"]), int(row["asp_end"])))
    agg_rows = [{"review_text": text, "asp_spans": sorted(spans)} for text, spans in grouped.items() if spans]
    return pd.DataFrame(agg_rows)

train_df_agg = aggregate_aspects(train_df)
val_df_agg = aggregate_aspects(val_df)
test_df_agg = aggregate_aspects(test_df)

print(f"\n[Aggregation] Train: {len(train_df)} individual rows -> {len(train_df_agg)} review unique")
print(f"[Aggregation] Val:   {len(val_df)} individual rows -> {len(val_df_agg)} review unique")
print(f"[Aggregation] Test:  {len(test_df)} individual rows -> {len(test_df_agg)} review unique")

### ASPECT PREPROCESS

In [ ]:
data_collator_aspect = DataCollatorForTokenClassification(tokenizer=tokenizer)

def preprocess_aspect(batch):
    tokenized = tokenizer(
        batch["review_text"], truncation=True, max_length=MAX_LENGTH, return_offsets_mapping=True
    )
    all_labels = []
    for i, offsets in enumerate(tokenized["offset_mapping"]):
        spans = batch["asp_spans"][i]
        labels = []
        for tok_start, tok_end in offsets:
            if tok_start == tok_end:
                labels.append(-100)
                continue
            tag = aspect_label2id["O"]
            for s, e in spans:
                if tok_start >= s and tok_end <= e:
                    tag = aspect_label2id["B-ASP"] if tok_start == s else aspect_label2id["I-ASP"]
                    break
            labels.append(tag)
        all_labels.append(labels)
    tokenized["labels"] = all_labels
    return tokenized

aspect_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df_agg[["review_text", "asp_spans"]]),
    "validation": Dataset.from_pandas(val_df_agg[["review_text", "asp_spans"]]),
    "test": Dataset.from_pandas(test_df_agg[["review_text", "asp_spans"]]),
})

aspect_ds = aspect_ds.map(preprocess_aspect, batched=True, load_from_cache_file=False)
aspect_ds = aspect_ds.remove_columns(
    [c for c in ["review_text", "asp_spans", "offset_mapping", "__index_level_0__"]
     if c in aspect_ds["train"].column_names]
)

flat_labels = [l for seq in aspect_ds["train"]["labels"] for l in seq if l != -100]
aspect_class_weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1, 2]), y=np.array(flat_labels)
)

aspect_class_weights = np.clip(aspect_class_weights, a_min=None, a_max=CLASS_WEIGHT_CAP)
aspect_class_weights_tensor = torch.tensor(aspect_class_weights, dtype=torch.float)
print("Aspect tag distribution weights (capped):", aspect_class_weights)

aspect_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

_sample = aspect_ds["train"][0]
_n_bi = sum(1 for l in _sample["labels"].tolist() if l in (1, 2))
print(f"[Sanity] نمونه اول train -> تعداد برچسب B/I: {_n_bi}")

### WEIGHTED FOCAL LOSS

In [ ]:
class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss * self.alpha[targets]).mean()


### MODEL: BERT + BiLSTM + CRF

In [ ]:
class BertBiLSTMCrfForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id,
                 use_crf=True, use_bilstm=False, class_weights=None,
                 emission_bias_strength=0.0, dropout_p=0.2):
        super().__init__()
        self.num_labels = num_labels
        self.use_crf = use_crf
        self.use_bilstm = use_bilstm
        self.emission_bias_strength = emission_bias_strength

        self.config = AutoConfig.from_pretrained(model_name, num_labels=num_labels,
                                                 id2label=id2label, label2id=label2id)
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout_p)

        hidden = self.config.hidden_size

        if use_bilstm:
            self.bilstm = nn.LSTM(hidden, hidden // 2, batch_first=True,
                                  bidirectional=True, dropout=0.1)
            self.classifier = nn.Linear(hidden, num_labels)
        else:
            self.bilstm = None
            self.classifier = nn.Linear(hidden, num_labels)

        self.crf = CRF(num_labels, batch_first=True) if use_crf else None
        self.class_weights = class_weights

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])

        if self.use_bilstm and self.bilstm is not None:
            sequence_output, _ = self.bilstm(sequence_output)

        emissions = self.classifier(sequence_output)

        if self.class_weights is not None and self.emission_bias_strength > 0:
            log_weights = torch.log(self.class_weights.to(emissions.device) + 1e-8)
            emissions = emissions + self.emission_bias_strength * log_weights.view(1, 1, -1)

        mask = attention_mask.bool()
        loss = None

        if self.use_crf:
            if labels is not None:
                crf_labels = labels.clone()
                crf_labels[crf_labels == -100] = 0
                loss = -self.crf(emissions, crf_labels, mask=mask, reduction="mean")

            decoded = self.crf.decode(emissions, mask=mask)
            pred_ids = torch.full(emissions.shape[:2], -100, dtype=torch.long, device=emissions.device)
            for i, seq in enumerate(decoded):
                pred_ids[i, :len(seq)] = torch.tensor(seq, device=emissions.device)
            logits_out = pred_ids
        else:
            weight = self.class_weights.to(emissions.device) if self.class_weights is not None else None
            loss_fct = WeightedFocalLoss(weight, gamma=1.5)
            active = labels.view(-1) != -100
            loss = loss_fct(emissions.view(-1, self.num_labels)[active],
                            labels.view(-1)[active])
            logits_out = emissions

        return TokenClassifierOutput(loss=loss, logits=logits_out)

aspect_model = BertBiLSTMCrfForTokenClassification(
    MODEL_NAME,
    num_labels=3,
    id2label=aspect_id2label,
    label2id=aspect_label2id,
    use_crf=USE_CRF,
    use_bilstm=USE_BILSTM,
    class_weights=aspect_class_weights_tensor,
    emission_bias_strength=0.3,
    dropout_p=0.2,
)

### METRICS

In [ ]:
def compute_metrics_aspect(pred):
    labels = pred.label_ids
    raw_preds = pred.predictions
    if isinstance(raw_preds, tuple):
        raw_preds = raw_preds[0]

    if raw_preds.ndim == 3 and raw_preds.shape[-1] > 1:
        preds = np.argmax(raw_preds, axis=-1)
    elif raw_preds.ndim == 3:
        preds = raw_preds.squeeze(-1)
    else:
        preds = raw_preds

    y_true, y_pred = [], []
    for true_seq, pred_seq in zip(labels, preds):
        true_tags, pred_tags = [], []
        for t, p in zip(true_seq, pred_seq):
            if t != -100:
                true_tags.append(aspect_id2label.get(int(t), "O"))
                p_int = int(p) if int(p) in aspect_id2label else 0
                pred_tags.append(aspect_id2label.get(p_int, "O"))
        if true_tags:
            y_true.append(true_tags)
            y_pred.append(pred_tags)

    return {
        "f1": seq_f1(y_true, y_pred),
        "f1_macro": seq_f1(y_true, y_pred, average="macro"),
        "precision": seq_precision(y_true, y_pred),
        "recall": seq_recall(y_true, y_pred),
    }

### TRAINER WITH LLRD

In [ ]:
class AspectTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            no_decay = ["bias", "LayerNorm.weight"]
            base_lr = self.args.learning_rate
            head_lr = base_lr * HEAD_LR_MULTIPLIER

            opt_params = []
            for i, layer in enumerate(self.model.bert.encoder.layer):
                lr = base_lr * (LLRD_DECAY ** (11 - i))
                for n, p in layer.named_parameters():
                    if not p.requires_grad:
                        continue
                    weight_decay = 0.0 if any(nd in n for nd in no_decay) else self.args.weight_decay
                    opt_params.append({"params": [p], "lr": lr, "weight_decay": weight_decay})

            for n, p in self.model.bert.named_parameters():
                if "encoder.layer" in n or not p.requires_grad:
                    continue
                weight_decay = 0.0 if any(nd in n for nd in no_decay) else self.args.weight_decay
                opt_params.append({"params": [p], "lr": base_lr * (LLRD_DECAY ** 12), "weight_decay": weight_decay})

            head_params = []
            for n, p in self.model.named_parameters():
                if n.startswith("bert."):
                    continue
                if p.requires_grad:
                    head_params.append(p)
            opt_params.append({"params": head_params, "lr": head_lr, "weight_decay": self.args.weight_decay})

            self.optimizer = torch.optim.AdamW(opt_params, lr=base_lr)
        return self.optimizer

    def _save(self, output_dir=None, state_dict=None):
        if state_dict is None:
            state_dict = self.model.state_dict()
        state_dict = {k: v.contiguous() if isinstance(v, torch.Tensor) else v for k, v in state_dict.items()}
        super()._save(output_dir, state_dict)


##### warmup calculation

In [ ]:
aspect_train_batch_size = 16 if not USE_BILSTM else 12
aspect_total_steps = (len(aspect_ds["train"]) // aspect_train_batch_size) * NUM_EPOCHS_ASPECT
aspect_warmup_steps = int(0.1 * aspect_total_steps)
print(f"[Aspect] Total steps: {aspect_total_steps} | Warmup steps: {aspect_warmup_steps}")

### Train Hyperparameters

In [ ]:
if os.path.exists(ASPECT_CKPT_DIR):
    shutil.rmtree(ASPECT_CKPT_DIR)

aspect_args = TrainingArguments(
    output_dir=ASPECT_CKPT_DIR,
    num_train_epochs=NUM_EPOCHS_ASPECT,
    per_device_train_batch_size=aspect_train_batch_size,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=aspect_warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.02,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=USE_FP16,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
)

aspect_trainer = AspectTrainer(
    model=aspect_model,
    args=aspect_args,
    train_dataset=aspect_ds["train"],
    eval_dataset=aspect_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_aspect,
    compute_metrics=compute_metrics_aspect,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)]
)


In [ ]:
print("\n=== CONFIG CHECK قبل از train ===")
print("MODEL_NAME:", MODEL_NAME)
print("tokenizer.name_or_path:", tokenizer.name_or_path)
print("HEAD_LR_MULTIPLIER:", HEAD_LR_MULTIPLIER)
print("emission_bias_strength:", aspect_model.emission_bias_strength)
print("weight_decay in args:", aspect_args.weight_decay)
print("dropout:", aspect_model.dropout.p)
print("USE_BILSTM:", USE_BILSTM)
print("trainer.args is aspect_args:", aspect_trainer.args is aspect_args)
print("trainer.model is aspect_model:", aspect_trainer.model is aspect_model)

print("\n--- Training Aspect Extraction (ParsBERT + BiLSTM + CRF + LLRD + Weighted, aggregated spans) ---")
aspect_trainer.train()

print("\nAspect Test Evaluation:")
print(aspect_trainer.evaluate(aspect_ds["test"]))


--- Loading Pars-ABSA ---
=== LOAD STATS (custom jsonl) ===
total_lines: 4421
json_decode_fail: 137
no_matching_key: 0
total_examples: 4284
no_text_or_aspects: 0
total_aspects: 4610
polarity_invalid: 0
term_not_found: 1
success: 4609
Custom synthetic aspects loaded: 4609
Final Dataset Splits (aspect-level, برای Sentiment) -> Train: 15137 | Val: 1251 | Test: 1250
Sentiment label distribution (train):
polarity
positive    0.643324
negative    0.213583
neutral     0.143093
Name: proportion, dtype: float64

[Aggregation] Train: 15137 ردیف تکی -> 9986 review منحصربه‌فرد
[Aggregation] Val:   1251 ردیف تکی -> 1117 review منحصربه‌فرد
[Aggregation] Test:  1250 ردیف تکی -> 1109 review منحصربه‌فرد


Map:   0%|          | 0/9986 [00:00<?, ? examples/s]

Map:   0%|          | 0/1117 [00:00<?, ? examples/s]

Map:   0%|          | 0/1109 [00:00<?, ? examples/s]

Aspect tag distribution weights (capped): [ 0.35144002  9.25091914 15.        ]
[Sanity] نمونه اول train -> تعداد برچسب B/I: 1


...

[Aspect] Total steps: 8320 | Warmup steps: 832

=== CONFIG CHECK قبل از train ===
MODEL_NAME: HooshvareLab/bert-base-parsbert-uncased
tokenizer.name_or_path: HooshvareLab/bert-base-parsbert-uncased
HEAD_LR_MULTIPLIER: 6
emission_bias_strength: 0.3
weight_decay in args: 0.02
dropout: 0.2
USE_BILSTM: True
trainer.args is aspect_args: True
trainer.model is aspect_model: True

--- Training Aspect Extraction (ParsBERT + BiLSTM + CRF + LLRD + Weighted, aggregated spans) ---


Epoch,Training Loss,Validation Loss,F1,F1 Macro,Precision,Recall
1,4.345367,5.788873,0.279506,0.279506,0.287467,0.271973
2,3.362081,5.224487,0.311955,0.311955,0.409152,0.252073
3,2.867531,6.720695,0.350031,0.350031,0.281187,0.463516
4,2.411624,7.846222,0.322347,0.322347,0.267505,0.405473
5,1.868216,9.489000,0.290934,0.290934,0.245714,0.356551
6,1.402974,11.486870,0.267909,0.267909,0.218668,0.345771



Aspect Test Evaluation:


Training Loss,Validation Loss,Epoch,F1,F1 Macro,Precision,Recall
1.402974,6.906610,6,0.338710,0.338710,0.271372,0.450495


{'eval_loss': 6.906610488891602, 'eval_f1': 0.33870967741935487, 'eval_f1_macro': 0.33870967741935487, 'eval_precision': 0.27137176938369784, 'eval_recall': 0.4504950495049505}


# SENTIMENT CLASSIFICATION

In [ ]:
def preprocess_sentiment(batch):
    tokenized = tokenizer(
        batch["review_text"], batch["term"],
        truncation=True, max_length=MAX_LENGTH, padding=False,
    )
    tokenized["labels"] = batch["label_id"]
    return tokenized

sentiment_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["review_text", "term", "label_id"]]),
    "validation": Dataset.from_pandas(val_df[["review_text", "term", "label_id"]]),
    "test": Dataset.from_pandas(test_df[["review_text", "term", "label_id"]]),
})
sentiment_ds = sentiment_ds.map(preprocess_sentiment, batched=True, load_from_cache_file=False)
sentiment_ds = sentiment_ds.remove_columns(
    [c for c in ["review_text", "term", "label_id", "__index_level_0__"]
     if c in sentiment_ds["train"].column_names]
)
sentiment_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator_sentiment = DataCollatorWithPadding(tokenizer=tokenizer)

sentiment_class_weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1, 2]), y=train_df["label_id"].values
)
sentiment_class_weights = np.clip(sentiment_class_weights, a_min=None, a_max=CLASS_WEIGHT_CAP)
sentiment_class_weights_tensor = torch.tensor(sentiment_class_weights, dtype=torch.float)
print("Sentiment class weights (capped):", sentiment_class_weights)

class BertForWeightedSentimentClassification(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id, class_weights=None, gamma=1.5):
        super().__init__()
        self.num_labels = num_labels
        self.config = AutoConfig.from_pretrained(
            model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
        )
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs[0][:, 0, :]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
            loss_fct = WeightedFocalLoss(weight, gamma=self.gamma)
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)

sentiment_model = BertForWeightedSentimentClassification(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id,
    class_weights=sentiment_class_weights_tensor, gamma=1.5,
)

def compute_metrics_sentiment(pred):
    labels = pred.label_ids
    logits = pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

class SentimentTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            no_decay = ["bias", "LayerNorm.weight"]
            base_lr = self.args.learning_rate
            head_lr = base_lr * HEAD_LR_MULTIPLIER

            opt_params = []
            for i, layer in enumerate(self.model.bert.encoder.layer):
                lr = base_lr * (LLRD_DECAY ** (11 - i))
                for n, p in layer.named_parameters():
                    if not p.requires_grad:
                        continue
                    weight_decay = 0.0 if any(nd in n for nd in no_decay) else self.args.weight_decay
                    opt_params.append({"params": [p], "lr": lr, "weight_decay": weight_decay})

            for n, p in self.model.bert.named_parameters():
                if "encoder.layer" in n or not p.requires_grad:
                    continue
                weight_decay = 0.0 if any(nd in n for nd in no_decay) else self.args.weight_decay
                opt_params.append({"params": [p], "lr": base_lr * (LLRD_DECAY ** 12), "weight_decay": weight_decay})

            head_params = [p for n, p in self.model.named_parameters() if not n.startswith("bert.") and p.requires_grad]
            opt_params.append({"params": head_params, "lr": head_lr, "weight_decay": self.args.weight_decay})

            self.optimizer = torch.optim.AdamW(opt_params, lr=base_lr)
        return self.optimizer

    def _save(self, output_dir=None, state_dict=None):
        if state_dict is None:
            state_dict = self.model.state_dict()
        state_dict = {k: v.contiguous() if isinstance(v, torch.Tensor) else v for k, v in state_dict.items()}
        super()._save(output_dir, state_dict)

sentiment_train_batch_size = 16
sentiment_total_steps = (len(sentiment_ds["train"]) // sentiment_train_batch_size) * NUM_EPOCHS_SENTIMENT
sentiment_warmup_steps = int(0.1 * sentiment_total_steps)
print(f"[Sentiment] Total steps: {sentiment_total_steps} | Warmup steps: {sentiment_warmup_steps}")

In [ ]:
if os.path.exists(SENTIMENT_CKPT_DIR):
    shutil.rmtree(SENTIMENT_CKPT_DIR)

sentiment_args = TrainingArguments(
    output_dir=SENTIMENT_CKPT_DIR,
    num_train_epochs=NUM_EPOCHS_SENTIMENT,
    per_device_train_batch_size=sentiment_train_batch_size,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=sentiment_warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.02,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    fp16=USE_FP16,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
)

sentiment_trainer = SentimentTrainer(
    model=sentiment_model,
    args=sentiment_args,
    train_dataset=sentiment_ds["train"],
    eval_dataset=sentiment_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_sentiment,
    compute_metrics=compute_metrics_sentiment,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("\n--- Training Sentiment Classification ---")
sentiment_trainer.train()

print("\nSentiment Test Evaluation:")
print(sentiment_trainer.evaluate(sentiment_ds["test"]))

# SAVE FINAL MODELS

In [ ]:
os.makedirs(ASPECT_FINAL_DIR, exist_ok=True)
os.makedirs(SENTIMENT_FINAL_DIR, exist_ok=True)

torch.save(aspect_model.state_dict(), os.path.join(ASPECT_FINAL_DIR, "pytorch_model.bin"))
aspect_model.config.save_pretrained(ASPECT_FINAL_DIR)
tokenizer.save_pretrained(ASPECT_FINAL_DIR)

torch.save(sentiment_model.state_dict(), os.path.join(SENTIMENT_FINAL_DIR, "pytorch_model.bin"))
sentiment_model.config.save_pretrained(SENTIMENT_FINAL_DIR)
tokenizer.save_pretrained(SENTIMENT_FINAL_DIR)

print("\n✅ Models saved in Drive sucssesfully! ")
print(" -", ASPECT_FINAL_DIR)
print(" -", SENTIMENT_FINAL_DIR)

Map:   0%|          | 0/15137 [00:00<?, ? examples/s]

Map:   0%|          | 0/1251 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Sentiment class weights (capped): [1.56067636 2.329486   0.51814199]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[Sentiment] Total steps: 7568 | Warmup steps: 756

--- Training Sentiment Classification ---


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
1,0.265986,0.353347,0.689049,0.686935,0.700072,0.715666,0.752658
2,0.205655,0.320454,0.852118,0.840296,0.853385,0.838580,0.845652
3,0.201967,0.331429,0.871303,0.861563,0.872098,0.858302,0.866873
4,0.053892,0.425015,0.853717,0.843115,0.854880,0.833815,0.855849
5,0.034162,0.454943,0.870504,0.857576,0.870787,0.854435,0.861041
6,0.020004,0.574651,0.875300,0.864077,0.875566,0.861331,0.866988
7,0.009103,0.602755,0.885691,0.874417,0.885559,0.878137,0.871166
8,0.021750,0.605623,0.886491,0.875315,0.886450,0.878350,0.872849



Sentiment Test Evaluation:


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
0.021750,0.664301,8,0.860000,0.844499,0.859758,0.846764,0.842335


{'eval_loss': 0.664300799369812, 'eval_accuracy': 0.86, 'eval_f1_macro': 0.8444992591768408, 'eval_f1_weighted': 0.8597580761764814, 'eval_precision_macro': 0.8467641639579889, 'eval_recall_macro': 0.8423354984523556}

✅ مدل‌ها با موفقیت در Drive ذخیره شدند:
 - /content/drive/MyDrive/Colab Notebooks/extracted/pars_absa_models/aspect_model
 - /content/drive/MyDrive/Colab Notebooks/extracted/pars_absa_models/sentiment_model


# Check Outputs

In [ ]:
from collections import Counter
preds_out = aspect_trainer.predict(aspect_ds["validation"])
raw = preds_out.predictions
flat_preds = raw.flatten() if raw.ndim == 2 else raw.squeeze(-1).flatten()
flat_labels = preds_out.label_ids.flatten()

valid_mask = flat_labels != -100
print("Pred distribution:", Counter(flat_preds[valid_mask].tolist()))
print("True distribution:", Counter(flat_labels[valid_mask].tolist()))

Pred distribution: Counter({0: 70773, 1: 1877, 2: 1565})
True distribution: Counter({0: 72061, 1: 1206, 2: 948})


In [ ]:
print(aspect_model.crf.transitions)
print(aspect_model.crf.start_transitions)

Parameter containing:
tensor([[ 0.0102,  0.0818, -0.1447],
        [-0.0157, -0.1030,  0.1919],
        [-0.1154,  0.0313, -0.0026]], device='cuda:0', requires_grad=True)
Parameter containing:
tensor([ 0.0120, -0.0767,  0.0276], device='cuda:0', requires_grad=True)


In [ ]:
count_no_aspect = sum(
    1 for labels in aspect_ds["train"]["labels"]
    if not any(l in (1, 2) for l in labels)
)
print(f"{count_no_aspect} / {len(aspect_ds['train'])} نمونه بدون هیچ B/I-ASP")

390 / 9986 نمونه بدون هیچ B/I-ASP


In [ ]:
print("HEAD_LR_MULTIPLIER:", HEAD_LR_MULTIPLIER)
print("emission_bias_strength:", aspect_model.emission_bias_strength)
print("weight_decay in args:", aspect_args.weight_decay)
print("dropout:", aspect_model.dropout.p)
print("Train size (with synthetic):", len(train_df))

HEAD_LR_MULTIPLIER: 6
emission_bias_strength: 0.3
weight_decay in args: 0.02
dropout: 0.2
Train size (with synthetic): 15137


In [ ]:
print("=== CONFIG CHECK ===")
print("HEAD_LR_MULTIPLIER:", HEAD_LR_MULTIPLIER)
print("emission_bias_strength:", aspect_model.emission_bias_strength)
print("weight_decay in args:", aspect_args.weight_decay)
print("dropout:", aspect_model.dropout.p)
print("Train size:", len(train_df))
print("aspect_class_weights:", aspect_class_weights)
print("trainer.args is aspect_args:", aspect_trainer.args is aspect_args)
print("trainer.model is aspect_model:", aspect_trainer.model is aspect_model)

=== CONFIG CHECK ===
HEAD_LR_MULTIPLIER: 6
emission_bias_strength: 0.3
weight_decay in args: 0.02
dropout: 0.2
Train size: 15137
aspect_class_weights: [ 0.35144002  9.25091914 15.        ]
trainer.args is aspect_args: True
trainer.model is aspect_model: True


In [ ]:
print("tokenizer.is_fast:", tokenizer.is_fast)

# همون نمونه idx=5
text = train_df.iloc[5]["review_text"]
start, end = train_df.iloc[5]["asp_start"], train_df.iloc[5]["asp_end"]
print("Aspect term (from span):", text[start:end])
print("asp_start, asp_end:", start, end)

enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_offsets_mapping=True)
print("First 20 offsets:", enc["offset_mapping"][:20])
print("Total tokens:", len(enc["offset_mapping"]))

tokenizer.is_fast: True
Aspect term (from span): کیفیت
asp_start, asp_end: 72 77
First 20 offsets: [(0, 0), (0, 2), (3, 7), (8, 11), (12, 16), (17, 19), (20, 24), (25, 30), (31, 34), (35, 40), (41, 43), (44, 49), (50, 55), (56, 59), (60, 63), (64, 66), (67, 71), (72, 77), (78, 82), (83, 88)]
Total tokens: 153


In [ ]:
sample = aspect_ds["train"][5]
print("Input ids length:", len(sample["input_ids"]))
print("Labels:", sample["labels"])
print("Number of non(-100) labels:", sum(1 for l in sample["labels"] if l != -100))
print("Number of B/I labels:", sum(1 for l in sample["labels"] if l in (1, 2)))

# و برای اطمینان بیشتر، توکن‌ها رو کنار لیبل‌ها ببینیم
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
for tok, lab in zip(tokens, sample["labels"]):
    lab_name = aspect_id2label.get(int(lab), "IGNORE(-100)") if int(lab) != -100 else "IGNORE(-100)"
    print(f"{tok:15s} -> {lab_name}")

Input ids length: 153
Labels: tensor([-100,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    1,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          

In [ ]:
dup_check = train_df.groupby("review_text")["term"].nunique()
multi_aspect_reviews = dup_check[dup_check > 1]
print(f"تعداد review‌های منحصربه‌فرد با بیش از یک aspect: {len(multi_aspect_reviews)}")
print(f"از کل {train_df['review_text'].nunique()} review منحصربه‌فرد")
print(multi_aspect_reviews.head(10))

تعداد review‌های منحصربه‌فرد با بیش از یک aspect: 2873
از کل 9986 review منحصربه‌فرد
review_text
(HD یعنی ۱۲۸۰ در ۷۲۰ پیکسل) و (Full HD یعنی ۱۹۲۰ در ۱۰۸۰ پیکسل) ۵ - ویدیوهای HD رو با پلیر خودش اجرا نمیکنه نقاط قوت: ۱ - سیستم‌عامل جدید آندروئید v ۴ ٫ ۱.۲ - Jelly Bean ۲ - نرم افزارهای قوی ۳ - طراحی بسیار زیبا و وزن سبک ۴ - صفحه نمایش بزرگ با رزولوشون نسبتا مناسب              5
(یعنی ۱۰ گرم سبک‌تر از iPad ۴) • صفحه نمایش و بلندگوهاگوشی PadFone ۲ دارای یک صفحه نمایش عریض ۴ ٫ ۷ اینچی از نوع Super IPS+ LCD بوده و از تکنولوژی IGZO کمپانی شارپ نیز بهره می‌برد که طبق ادعای این کمپانی، انرژی کمتری مصرف کرده و طول عمر بالاتری نسبت به صفحات LCD معمولی دارد.    2
. با توجه به قیمت آن گوشی خیلی خوبی است فقط اگه یه اسپیکر کوچک داشت بهتر بود.                                                                                                                                                                                                          2
Canon LiDE ۱۰۰ اسکنری سبک‌وزن و نسبتا جمع و جور می‌باشد که ا